# Neural Pandora's Cluster using REGenerative AutoEncoders (REGAE)

#### Basic imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

torch.autograd.set_detect_anomaly(False)

import numpy as np
import random
import pickle
torch.__version__

#### Basic functions

In [ ]:
import os
import requests
from urllib.parse import urlparse

def download_file(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, 'wb') as file:
            file.write(response.content)
        print(f"File downloaded successfully as {filename}")
    else:
        print(f"Failed to download file from {url}")

def check_or_download_file(file_url, file_filename):
    if os.path.exists(file_filename):
        print(f"{file_filename} already exists in the current folder.")
    else:
        print(f"{file_filename} not found. Downloading from {file_url}...")
        download_file(file_url, file_filename)

In [ ]:
from torchvision import transforms
import cv2

def load_image(image_path):
    # Read the RGB image using OpenCV
    image = cv2.imread(image_path)
    # OpenCV reads images in BGR format, so we need to convert it to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image

def image_to_tensor(image):
    # Convert the image to a torch tensor
    transform = transforms.ToTensor()
    tensor = transform(image)
    return tensor

In [ ]:
def count_parameters(network):
    return sum(p.numel() for p in network.parameters() if p.requires_grad)

#### Loading image

In [ ]:
file_url = "https://stsci-opo.org/STScI-01GQQF9WVPFVMCVHRZY54N2TAR.png"
parsed_url = urlparse(file_url)
file_filename = parsed_url.path[1:]

check_or_download_file(file_url, file_filename)

In [ ]:
rgb_image = load_image(file_filename)
tensor_image = image_to_tensor(rgb_image)
tensor_image_shape = tensor_image.shape

# tensor_image_mean = tensor_image.mean(dim=(1, 2), keepdim=True)
# tensor_image_std = tensor_image.std(dim=(1, 2), keepdim=True)
# tensor_image = (tensor_image - tensor_image_mean) / tensor_image_std

tensor_image = 2 * tensor_image - 1

tensor_image.requires_grad_(False)

tensor_image_shape


In [ ]:
patch_size = 32
compressed_shape = tuple((s // patch_size) for s in tensor_image_shape[1:])

n_pixels = int(np.prod(tensor_image_shape))
n_patches = int(np.prod(compressed_shape))
n_pixels, n_patches, n_pixels / n_patches

#### Image patch functions

In [ ]:
def get_image_patch(tensor_image, patch_size, x, y):
    assert 0. <= x < 1. and 0. <= y < 1.
    tensor_image_shape = tensor_image.shape
    img_x = int(x * (tensor_image_shape[1] - patch_size))
    img_y = int(y * (tensor_image_shape[2] - patch_size))
    image_patch = tensor_image[:, img_x:img_x + patch_size, img_y:img_y + patch_size]

    return image_patch.to(device)

In [ ]:
image_patch = get_image_patch(tensor_image, patch_size, 0.42, 0.42)
image_patch_shape = image_patch.shape

print(image_patch_shape)
del image_patch

In [ ]:
def get_random_patch_batch(batch_size, tensor_image, image_patch_shape=image_patch_shape, patch_size=patch_size):
    image_patch_batch = torch.empty((batch_size,) + image_patch_shape, device=device)
    xs = torch.rand(batch_size, device=device)
    ys = torch.rand(batch_size, device=device)
    for b in range(batch_size):
        image_patch_batch[b] = get_image_patch(tensor_image, patch_size, xs[b].item(), ys[b].item())
    
    return image_patch_batch
    

In [ ]:
image_patch_batch = get_random_patch_batch(1, tensor_image)

print(image_patch_batch.shape)
del image_patch_batch

In [ ]:
def get_patch_any_size(tensor_image, x_start, x_end, y_start, y_end, tensor_image_shape=tensor_image_shape, patch_size=patch_size):
    image_patch_l = []
    for x in range(x_start, x_end, patch_size):
        for y in range(y_start, y_end, patch_size):
            image_patch = get_image_patch(tensor_image, patch_size, x / tensor_image_shape[1], y / tensor_image_shape[2])
            image_patch_l.append(image_patch)
    
    return torch.stack(image_patch_l)

In [ ]:
def reconstruct_image_from_patch_batch(image_patch, rows, cols, channels=3):
    image_patch = image_patch.permute(1, 2, 0, 3)
    image_patch = image_patch.reshape(channels, patch_size, rows, cols * patch_size)
    image_patch = image_patch.permute(0, 2, 1, 3)
    image_patch = image_patch.reshape(channels, rows * patch_size, cols * patch_size)
    return image_patch

In [ ]:
def requantize(x):
    qx = x.clamp_(-1, 1).add_(1).mul_(255 / 2).int().clamp_(0, 255)
    return qx.type_as(x).mul_(2 / 255).add_(-1)

## Layers and Networks

In [ ]:
class MHAttention(nn.Module):
    def __init__(self, fin, fhidden=None, nheads=8, dim=1, dropout=0, residual=False, condition=False, lat_size=0):
        super(MHAttention, self).__init__()

        self.fin = fin
        self.fhidden = fin if fhidden is None else fhidden
        self.nheads = nheads
        self.dim = dim
        self.residual = residual
        self.condition = condition

        if self.dim == 1:
            conv_fn = nn.Conv1d
        elif self.dim == 2:
            conv_fn = nn.Conv2d
        elif self.dim == 3:
            conv_fn = nn.Conv3d
        else:
            raise RuntimeError('Only 1, 2 and 3 dimensions are supported. Received {}.'.format(dim))

        self.q = conv_fn(self.fin, self.nheads * self.fhidden, 1, 1, 0)
        self.k = conv_fn(self.fin, self.nheads * self.fhidden, 1, 1, 0)
        self.v = conv_fn(self.fin, self.nheads * self.fhidden, 1, 1, 0)
        self.conv_out = conv_fn(self.nheads * self.fhidden, self.fin, 1, 1, 0)

        if self.condition:
            self.q_mod = nn.Linear(lat_size, self.nheads * self.fhidden * 2)
            self.k_mod = nn.Linear(lat_size, self.nheads * self.fhidden * 2)
            self.v_mod = nn.Linear(lat_size, self.nheads * self.fhidden * 2)
            self.conv_out_mod = nn.Linear(lat_size, self.fin * 2)

        self.temp = (self.fhidden // nheads) ** -0.5
        self.dropout = None
        if dropout > 0:
            self.dropout = nn.Dropout(dropout)

    def forward(self, x, lat=None):
        assert (lat is not None) == self.condition
        batch_size = x.shape[0]

        x_q = self.q(x).view(batch_size,  self.nheads, self.fhidden, -1).view(batch_size * self.nheads, self.fhidden, -1).contiguous()
        x_k = self.k(x).view(batch_size,  self.nheads, self.fhidden, -1).view(batch_size * self.nheads, self.fhidden, -1).contiguous()
        x_v = self.v(x).view(batch_size,  self.nheads, self.fhidden, -1).view(batch_size * self.nheads, self.fhidden, -1).contiguous()

        if self.condition:
            x_q_mod_m, x_q_mod_a = torch.chunk(self.q_mod(lat).view(batch_size,  self.nheads, self.fhidden * 2).reshape(batch_size * self.nheads, self.fhidden * 2, 1), 2, 1)
            x_q = x_q + x_q_mod_m * x_q + x_q_mod_a
            x_k_mod_m, x_k_mod_a = torch.chunk(self.k_mod(lat).view(batch_size,  self.nheads, self.fhidden * 2).reshape(batch_size * self.nheads, self.fhidden * 2, 1), 2, 1)
            x_k = x_k + x_k_mod_m * x_k + x_k_mod_a
            x_v_mod_m, x_v_mod_a = torch.chunk(self.v_mod(lat).view(batch_size,  self.nheads, self.fhidden * 2).reshape(batch_size * self.nheads, self.fhidden * 2, 1), 2, 1)
            x_v = x_v + x_v_mod_m * x_v + x_v_mod_a

        attn = torch.bmm(x_q.permute(0, 2, 1), x_k) * self.temp
        attn = torch.softmax(attn, dim=2)
        if self.dropout is not None:
            attn = self.dropout(attn)
        x_out = torch.bmm(attn, x_v.permute(0, 2, 1))

        x_out = x_out.permute(0, 2, 1).view(batch_size, self.nheads, self.fhidden, -1).contiguous().view(batch_size, self.nheads * self.fhidden, -1).contiguous()
        x_out = x_out.view(*([batch_size, self.nheads * self.fhidden] + [s for s in x.shape[2:]]))

        x_out = self.conv_out(x_out)

        if self.condition:
            x_out_mod_m, x_out_mod_a = torch.chunk(self.conv_out_mod(lat).view(*([batch_size, self.fin * 2] + [1 for _ in range(self.dim)])), 2, 1)
            x_out = x_out + x_out_mod_m * x_out + x_out_mod_a

        if self.residual:
            return x + x_out
        else:
            return x_out


In [ ]:
class LinearBlock(nn.Module):
    def __init__(self, fin, fout, fhidden=None, bias=True):
        super(LinearBlock, self).__init__()

        self.fin = fin
        self.fout = fout
        self.fhidden = max(fin + fout, 1) if fhidden is None else fhidden

        self.block = nn.Sequential(
            nn.Linear(self.fin, self.fhidden, bias=bias),
            nn.SiLU(True),
            nn.Linear(self.fhidden, self.fout, bias=bias)
        )
        self.shortcut = nn.Linear(self.fin, self.fout, bias=bias)

    def forward(self, x):
        return self.shortcut(x) + self.block(x)
    

In [ ]:
class DynaConv(nn.Module):
    def __init__(self, lat_size, fin, fout, kernel_size=1, stride=1, padding=0, groups=1, dim=2, lat_factor=1):
        super(DynaConv, self).__init__()

        self.lat_size = lat_size
        self.fin = fin
        self.fout = fout
        self.dim = dim

        if dim == 1:
            self.conv_f = F.conv1d
        elif dim == 2:
            self.conv_f = F.conv2d
        elif dim == 3:
            self.conv_f = F.conv3d
        else:
            raise RuntimeError('Only 1, 2 and 3 dimensions are supported. Received {}.'.format(dim))

        self.k_size = self.fout * (self.fin // groups) * (kernel_size ** dim)
        self.b_size = self.fout

        if lat_factor == 0:
            self.dyna_k = nn.Linear(self.lat_size, self.k_size + self.b_size)
        else:
            self.dyna_k = LinearBlock(self.lat_size, self.k_size + self.b_size, int(self.lat_size * lat_factor))

        self.prev_lat = None
        self.k, self.b = None, 0
        self.kernel_size = [kernel_size for _ in range(self.dim)]
        self.stride = stride
        self.padding = padding
        self.groups = groups

    def forward(self, x, lat):
        batch_size = x.size(0)

        if self.prev_lat is None or self.prev_lat.data_ptr() != lat.data_ptr():
            ks = self.dyna_k(lat)
            k, b = torch.split(ks, [self.k_size, self.b_size], dim=1)
            self.k = k.view([batch_size, self.fout, self.fin // self.groups] + self.kernel_size)
            self.k = self.k.reshape([batch_size * self.fout, self.fin // self.groups] + self.kernel_size)

            self.b = b.view([batch_size, self.fout]).reshape([1, batch_size * self.fout] + [1 for _ in range(self.dim)])

            self.prev_lat = lat

        x_new = x.reshape([1, batch_size * self.fin] + [x.size(d + 2) for d in range(self.dim)])
        x_new = self.conv_f(x_new, self.k, stride=self.stride, padding=self.padding, groups=batch_size * self.groups) + self.b
        x_new = x_new.reshape([batch_size, self.fout] + [x.size(d + 2) for d in range(self.dim)])

        return x_new
    

In [ ]:
class GlobalNorm(nn.Module):
    def __init__(self, fin, dim=2):
        super(GlobalNorm, self).__init__()
        self.fin = fin
        self.dim = dim
        if self.dim == 1:
            conv_mod = nn.Conv1d
        elif self.dim == 2:
            conv_mod = nn.Conv2d
        elif self.dim == 3:
            conv_mod = nn.Conv3d
        else:
            raise RuntimeError('Only 1, 2 and 3 dimensions are supported. Received {}.'.format(dim))

        self.global_norm = nn.GroupNorm(1, fin, affine=False)
        self.mix_conv = conv_mod(fin, fin, 1, 1, 0)

    def forward(self, x):
        return self.mix_conv(self.global_norm(x))


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, fin, fout, fhidden=None, dim=2, pos_enc=False, image_size=0, condition=False, lat_size=0, attention=False, attn_patch_size=1, normalize=True):
        super(ConvBlock, self).__init__()

        self.fin = fin
        self.fout = fout

        self.dim = dim
        if dim == 1:
            conv_mod = nn.Conv1d
            convtr_mod = nn.ConvTranspose1d
        elif dim == 2:
            conv_mod = nn.Conv2d
            convtr_mod = nn.ConvTranspose2d
        elif dim == 3:
            conv_mod = nn.Conv3d
            convtr_mod = nn.ConvTranspose3d
        else:
            raise RuntimeError('Only 1, 2 and 3 dimensions are supported. Received {}.'.format(dim))

        self.fhidden = max(fin + fout, 1) if fhidden is None else fhidden

        self.pos_enc = pos_enc
        self.image_size = image_size
        self.condition = condition
        self.lat_size = lat_size
        self.attention = attention
        self.attn_patch_size = max(attn_patch_size, 1)
        self.normalize = normalize

        self.conv_wave = conv_mod(self.fin, self.fhidden, kernel_size=3, stride=1, padding=1)
        if self.normalize:
            self.norm_wave = GlobalNorm(self.fhidden, dim=dim)

        if self.condition:
            self.wave_cond = LinearBlock(self.lat_size, self.fhidden * 2)
            # self.wave_cond = DynaConv(self.lat_size, self.fhidden, self.fhidden)
            # if self.normalize:
            #     self.norm_cond = GlobalNorm(self.fhidden, dim=dim)

        if pos_enc:
            self.wave_pos = nn.Parameter(torch.randn([1, self.fhidden] + [self.image_size for _ in range(self.dim)]))
            # if self.condition:
            #     self.wave_pos_cond = nn.Linear(self.lat_size, self.fhidden * 2)
            # if self.normalize:
            #     self.norm_pos = GlobalNorm(self.fhidden, dim=dim)

        if self.attention:
            assert self.image_size % self.attn_patch_size == 0

            self.fattn = int(self.fhidden * (self.attn_patch_size / 4))

            self.wave_attn = MHAttention(self.fattn, dim=dim) # , condition=condition, lat_size=lat_size
            
            # if self.patch_size > 1:
            self.wave_to_patch = conv_mod(self.fhidden, self.fattn, self.attn_patch_size, self.attn_patch_size, 0)
            self.patch_to_wave = convtr_mod(self.fattn, self.fhidden, self.attn_patch_size, self.attn_patch_size, 0)

            # if self.normalize:
            #     self.norm_attn = GlobalNorm(self.fhidden, dim=dim)

        self.nl_conv = nn.Sequential(
            conv_mod(self.fhidden, self.fhidden, kernel_size=1, stride=1, padding=0),
            nn.GELU(),
            conv_mod(self.fhidden, self.fout, kernel_size=1, stride=1, padding=0)
        )

        self.l_conv = conv_mod(self.fhidden, self.fout, kernel_size=1, stride=1, padding=0)

    def forward(self, x, lat=None):
        assert (lat is None) == (not self.condition)

        x_wave = self.conv_wave(x)
        if self.normalize:
            x_wave = self.norm_wave(x_wave)

        if self.condition:
            x_wave_mod_m, x_wave_mod_a = torch.chunk(self.wave_cond(lat).view(*([x.shape[0], self.fhidden * 2] + [1 for _ in range(self.dim)])), 2, 1)
            x_wave = x_wave + x_wave_mod_m * x_wave + x_wave_mod_a
            # x_wave = self.wave_cond(x_wave, lat)
            # if self.normalize:
            #     x_wave = self.norm_cond(x_wave)

        if self.pos_enc:
            x_wave_pos = self.wave_pos.repeat([x.shape[0], 1] + [1 for _ in range(self.dim)])
            if np.any([self.wave_pos.shape[2:] != x_wave.shape[2:]]):
                x_wave_pos = F.interpolate(x_wave_pos, size=x_wave.shape[2:], mode='bicubic')
            
            # if self.condition:
            #     x_wave_pos_mod_m, x_wave_pos_mod_a = torch.chunk(self.wave_pos_cond(lat).view(*([x.shape[0], self.fhidden * 2] + [1 for _ in range(self.dim)])), 2, 1)
            #     x_wave_pos = x_wave_pos + x_wave_pos_mod_m * x_wave_pos + x_wave_pos_mod_a

            x_wave = x_wave + x_wave_pos
            # if self.normalize:
            #     x_wave = self.norm_pos(x_wave)

        if self.attention:
            x_wave_patch = x_wave
            #if self.attn_patch_size > 1:
            x_wave_patch = self.wave_to_patch(x_wave_patch)

            # if self.condition:
            #     x_wave_patch = self.wave_attn(x_wave_patch, lat)
            # else:
            x_wave_patch = self.wave_attn(x_wave_patch)

            #if self.attn_patch_size > 1:
            x_wave_patch = self.patch_to_wave(x_wave_patch)

            x_wave = x_wave + x_wave_patch
            # if self.normalize:
            #     x_wave = self.norm_attn(x_wave)

        x_wave = self.l_conv(x_wave) + self.nl_conv(x_wave)

        return x_wave
    


In [ ]:
class ConvU(nn.Module):
    def __init__(self, fin, fout, fhidden, image_size, enc_position=True, attention=True, n_condition=0, lat_size=0, normalize=True, dim=2):
        super().__init__()
        self.fin = fin
        self.fout = fout
        self.fhidden = fhidden
        self.image_size = image_size
        self.n_layers = int(np.log2(image_size)) - 1
        self.enc_position = enc_position
        self.attention = attention
        self.n_condition = n_condition
        self.lat_size = lat_size
        self.dim = dim

        if self.dim == 1:
            conv_mod = nn.Conv1d
        elif self.dim == 2:
            conv_mod = nn.Conv2d
        elif self.dim == 3:
            conv_mod = nn.Conv3d
        else:
            raise RuntimeError('Only 1, 2 and 3 dimensions are supported. Received {}.'.format(dim))

        self.f_size = np.concatenate([np.linspace(fin, fhidden, self.n_layers), np.linspace(fhidden, fout, self.n_layers + 1)]).astype(int).tolist()
        l_size = (np.array(self.f_size[:-1]) * np.array(self.f_size[1:]))
        self.l_size = ((self.lat_size // self.n_layers) + ((l_size / np.sum(l_size)) ** 0.5) * self.lat_size).astype(int).tolist()

        for d in range(self.n_condition):
            self.register_module('cond_fc_%d'%d, nn.Sequential(
                LinearBlock(self.lat_size, self.lat_size * 2),
                LinearBlock(self.lat_size * 2, np.sum(self.l_size))
            ))
            self.register_module('cond_down_convs_%d' % d, nn.ModuleList([
                ConvBlock(self.f_size[l], self.f_size[l], dim=dim, condition=True, lat_size=self.l_size[l], normalize=normalize)
            for l in range(self.n_layers)]))
            self.register_module('cond_up_convs_%d' % d, nn.ModuleList([
                ConvBlock(self.f_size[l], self.f_size[l], dim=dim, condition=True, lat_size=self.l_size[l], normalize=normalize)
            for l in range(self.n_layers, self.n_layers * 2)]))

        self.down_convs = nn.ModuleList([
            nn.Sequential(
                ConvBlock(self.f_size[l], self.f_size[l + 1], dim=dim,
                              pos_enc=self.enc_position, image_size=self.image_size // 2 ** l, 
                              attention=self.attention, attn_patch_size=(self.image_size // 2 ** l) // 8, normalize=normalize),
                ConvBlock(self.f_size[l + 1], self.f_size[l + 1], dim=dim,
                              pos_enc=self.enc_position, image_size=self.image_size // 2 ** l, normalize=normalize),
            )
        for l in range(self.n_layers)])

        # TODO: add residual frobinator here

        self.inject_convs = nn.ModuleList([
            conv_mod(self.f_size[self.n_layers + (self.n_layers - l)], self.f_size[l], 1, 1, 0)
        for l in range(self.n_layers + 1, self.n_layers * 2)])
        self.up_convs = nn.ModuleList([
            nn.Sequential(
                ConvBlock(self.f_size[l], self.f_size[l + 1], dim=dim,
                              pos_enc=self.enc_position, image_size=self.image_size // 2 ** ((self.n_layers * 2) - 1 - l),
                              attention=self.attention, attn_patch_size=(self.image_size // 2 ** ((self.n_layers * 2) - 1 - l)) // 8, normalize=normalize),
                ConvBlock(self.f_size[l + 1], self.f_size[l + 1], dim=dim,
                              pos_enc=self.enc_position, image_size=self.image_size // 2 ** ((self.n_layers * 2) - 1 - l), normalize=normalize),
            )
        for l in range(self.n_layers, self.n_layers * 2)])

    def forward(self, x, lat=None):
        # assert (lat is None) == (not self.condition)

        cond_lat = []
        if not lat is None:
            for d in range(self.n_condition):
                if not lat[d] is None:
                    cond_lat_tmp = self.get_submodule('cond_fc_%d' % d)(lat[d])
                    cond_lat.append(torch.split(cond_lat_tmp, self.l_size, 1))
                else:
                    cond_lat.append(None)

        x_l = []
        new_x = x
        for l in range(self.n_layers):
            if not lat is None:
                for d in range(self.n_condition):
                    if not cond_lat[d] is None:
                        new_x = self.get_submodule('cond_down_convs_%d' % d)[l](new_x, cond_lat[d][l])

            new_x = self.down_convs[l](new_x)

            if l < self.n_layers - 1:
                x_l.append(new_x)
                new_x = F.interpolate(new_x, scale_factor=0.5, mode='nearest')

        for l in range(self.n_layers):
            if l > 0:
                new_x = new_x + self.inject_convs[l - 1](x_l[self.n_layers - 1 - l])
            
            if not lat is None:
                for d in range(self.n_condition):
                    if not cond_lat[d] is None:
                        new_x = self.get_submodule('cond_up_convs_%d' % d)[l](new_x, cond_lat[d][l + self.n_layers])

            new_x = self.up_convs[l](new_x)

            if l < self.n_layers - 1:
                new_x = F.interpolate(new_x, scale_factor=2., mode='nearest')

        return new_x

In [ ]:
class ConvStack(nn.Module):
    def __init__(self, fin, fout, n_layers, image_size=0, enc_position=False, attention=False, n_condition=0, lat_size=0, scale_factor=1., normalize=True, dim=2):
        super(ConvStack, self).__init__()
        self.fin = fin
        self.n_layers = n_layers
        self.image_size = image_size
        self.enc_position = enc_position
        self.attention = attention
        self.n_condition = n_condition
        self.lat_size = lat_size
        self.scale_factor = scale_factor

        self.f_size = np.linspace(fin, fout, n_layers + 1).astype(int).tolist()
        l_size = (np.array(self.f_size[:-1]) * np.array(self.f_size[1:]))
        self.l_size = ((self.lat_size // n_layers) + ((l_size / np.sum(l_size)) ** 0.5) * self.lat_size).astype(int).tolist()

        for d in range(self.n_condition):
            self.register_module('cond_fc_%d'%d, nn.Sequential(
                LinearBlock(self.lat_size, self.lat_size * 2),
                LinearBlock(self.lat_size * 2, np.sum(self.l_size))
            ))
            self.register_module('cond_conv_%d' % d, nn.ModuleList([
                ConvBlock(self.f_size[l], self.f_size[l], dim=dim, condition=True, lat_size=self.l_size[l], normalize=normalize)
            for l in range(self.n_layers)]))

        self.res_convs = nn.ModuleList([
            nn.Sequential(
                ConvBlock(self.f_size[l], self.f_size[l + 1], dim=dim,
                              pos_enc=self.enc_position, image_size=int(self.image_size * (scale_factor ** l)),
                              attention=self.attention, attn_patch_size=int(self.image_size * (scale_factor ** l)) // 8, normalize=normalize),
                ConvBlock(self.f_size[l + 1], self.f_size[l + 1], dim=dim,
                              pos_enc=self.enc_position, image_size=int(self.image_size * (scale_factor ** l)), normalize=normalize)
            )
        for l in range(self.n_layers)])


    def forward(self, x, lat=None):
        # assert (lat is None) == (not self.condition)

        cond_lat = []
        if not lat is None:
            for d in range(self.n_condition):
                if not lat[d] is None:
                    cond_lat_tmp = self.get_submodule('cond_fc_%d' % d)(lat[d])
                    cond_lat.append(torch.split(cond_lat_tmp, self.l_size, 1))
                else:
                    cond_lat.append(None)

        new_x = x
        for l in range(self.n_layers):
            if not lat is None:
                for d in range(self.n_condition):
                    if not cond_lat[d] is None:
                        new_x = self.get_submodule('cond_conv_%d' % d)[l](new_x, cond_lat[d][l])
            
            new_x = self.res_convs[l](new_x)
            if self.scale_factor != 1.:
                new_x = F.interpolate(new_x, scale_factor=self.scale_factor, mode='nearest')
        
        return new_x


In [ ]:
class IRMLinear(nn.Module):
    def __init__(self, fin, n_layers=4):
        super(IRMLinear, self).__init__()
        self.fin = fin
        self.block = nn.Sequential(*[nn.Linear(fin, fin, bias=False) for _ in range(n_layers)])
        # for l in self.block:
        #     nn.init.normal_(l.weight, 0, 0.5 / self.fin ** 0.5)
        self.compressed_block = None

    def forward(self, x):
        if self.training:
            self.compressed_block = None
            return self.block(x)
        else:
            if self.compressed_block is None:
                with torch.no_grad():
                    compressed_block = self.block[0].weight.t()
                    for l in self.block[1:]:
                        compressed_block = compressed_block @ l.weight.t()
                    self.compressed_block = compressed_block.t()
            return F.linear(x, self.compressed_block)

In [ ]:
class IRMConv(nn.Module):
    def __init__(self, fin, fout, n_layers=4, dim=2, kernel_size=3, stride=1, padding=1):
        super(IRMConv, self).__init__()
        self.fin = fin
        self.dim = dim
        if self.dim == 1:
            conv_mod = nn.Conv1d
        elif self.dim == 2:
            conv_mod = nn.Conv2d
        elif self.dim == 3:
            conv_mod = nn.Conv3d
        else:
            raise RuntimeError('Only 1, 2 and 3 dimensions are supported. Received {}.'.format(dim))

        self.f_size = np.linspace(fin, fout, n_layers + 1).astype(int).tolist()

        self.block = nn.Sequential(*[conv_mod(self.f_size[l], self.f_size[l + 1], kernel_size, stride, padding, bias=False) for l in range(n_layers)])
        # for l in self.block:
        #     nn.init.normal_(l.weight, 0, 0.5 / self.fin ** 0.5)
        self.compressed_block = None

    def forward(self, x):
        return self.block(x)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, channels, n_filter, image_size, lat_size, norm_out=False, **kwargs):
        super(Encoder, self).__init__()

        self.channels = channels
        self.n_filter = n_filter
        self.n_layers = int(np.log2(image_size)) - 1
        self.image_size = image_size
        self.lat_size = lat_size
        self.norm_out = norm_out
        
        self.inlet = IRMConv(self.channels, self.n_filter)
        self.norm_stack = ConvStack(self.n_filter, self.n_filter * 2, self.n_layers // 2,
                                    self.image_size, True, False, 0, 0, 0.5)
        self.denorm_stack = ConvStack(self.n_filter * 2, self.n_filter * 4, self.n_layers - (self.n_layers // 2),
                                      self.image_size, True, False, 0, 0, 0.5, normalize=False)
        self.out_reduce = LinearBlock(self.n_filter * 4 * 4, self.lat_size)

    def forward(self, image_patch):
        
        lat_code = self.inlet(image_patch)
        lat_code = self.norm_stack(lat_code)
        lat_code = self.denorm_stack(lat_code)
        lat_code = self.out_reduce(lat_code.view(lat_code.shape[0], -1))
        
        if self.norm_out:
            lat_code = F.normalize(lat_code)

        return lat_code

In [ ]:
class Decoder(nn.Module):
    def __init__(self, channels, n_filter, image_size, n_condition, lat_size, gravity=True, dim=2, irm_kernel_size=3, irm_stride=1, irm_padding=1, n_filter_factor=4, **kwargs):
        super(Decoder, self).__init__()

        self.channels = channels
        self.n_filter = n_filter
        self.image_size = image_size
        self.n_condition = n_condition
        self.lat_size = lat_size
        self.gravity = gravity

        self.dim = dim
        if self.dim == 1:
            conv_mod = nn.Conv1d
        elif self.dim == 2:
            conv_mod = nn.Conv2d
        elif self.dim == 3:
            conv_mod = nn.Conv3d
        else:
            raise RuntimeError('Only 1, 2 and 3 dimensions are supported. Received {}.'.format(dim))
        
        self.inlet = IRMConv(self.channels, self.n_filter, 4, dim, irm_kernel_size, irm_stride, irm_padding)
        self.norm_U = ConvU(self.n_filter, self.n_filter * (2 if gravity else 1), int(self.n_filter * n_filter_factor),
                            self.image_size, True, True, self.n_condition, self.lat_size, dim=dim)
        # TODO: add sparcifying activations here
        self.denorm_U = ConvU(self.n_filter, self.n_filter, int(self.n_filter * n_filter_factor),
                              image_size, True, True, self.n_condition, self.lat_size, normalize=False, dim=dim)
        self.out_conv = conv_mod(self.n_filter, self.channels, 1, 1, 0)

    def forward(self, image_init, lat_codes=None):

        image_out = image_init

        if image_init.ndim == 2:
            image_out = image_out[:, None, :]

        image_out = self.inlet(image_out)
        image_out = self.norm_U(image_out, lat_codes)

        if self.gravity:
            image_out, image_out_g = torch.chunk(image_out, 2, 1)
            # image_out = image_out * (torch.relu(image_out_g + 1) + 1e-4)
            image_out = image_out * torch.exp(image_out_g)

        image_out = self.denorm_U(image_out, lat_codes)
        image_out = self.out_conv(image_out)

        if image_init.ndim == 2:
            image_out = image_out.squeeze(1)

        return image_out

In [ ]:
class VGGPerceptualLoss(nn.Module):
    DEFAULT_FEATURE_LAYERS = [0, 1, 2, 3]
    IMAGENET_RESIZE = (patch_size * 2, patch_size * 2) #(224, 224)
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]
    IMAGENET_SHAPE = (1, 3, 1, 1)

    def __init__(self, resize=True, feature_layers=None, style_layers=None):
        super().__init__()
        self.resize = resize
        self.feature_layers = feature_layers or self.DEFAULT_FEATURE_LAYERS
        self.style_layers = style_layers or []
        features = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT).features
        self.blocks = nn.ModuleList([
            features[:4].eval(),
            features[4:9].eval(),
            features[9:16].eval(),
            features[16:23].eval(),
        ])
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer("mean", torch.tensor(self.IMAGENET_MEAN).view(self.IMAGENET_SHAPE))
        self.register_buffer("std", torch.tensor(self.IMAGENET_STD).view(self.IMAGENET_SHAPE))

    def _transform(self, tensor):
        # if tensor.shape != self.IMAGENET_SHAPE:
        #     tensor = tensor.repeat(self.IMAGENET_SHAPE)
        tensor = (tensor + 1.) / 2
        tensor = (tensor - self.mean) / self.std
        if self.resize:
            tensor = nn.functional.interpolate(tensor, mode='bilinear', size=self.IMAGENET_RESIZE, align_corners=False)
        return tensor

    def _calculate_gram(self, tensor):
        act = tensor.reshape(tensor.shape[0], tensor.shape[1], -1)
        return act @ act.permute(0, 2, 1)

    def forward(self, output, target):
        output, target = self._transform(output), self._transform(target)
        loss = 0.
        for i, block in enumerate(self.blocks):
            output, target = block(output), block(target)
            if i in self.feature_layers:
                loss += nn.functional.l1_loss(output, target)
            if i in self.style_layers:
                gram_output, gram_target = self._calculate_gram(output), self._calculate_gram(target)
                loss += nn.functional.l1_loss(gram_output, gram_target)
        return loss
    

## Noise model definition

In [ ]:
# Noise distribution
def randinit(shape):
    return F.group_norm(torch.randn(shape, device=device), 1)

# Number of recurrent calls for the model
n_calls = 7

def blend_factor(call_idx, ministep=0, n_ministeps=1000, reverse=False, res_value=0.0586, n_calls=n_calls):
    if isinstance(call_idx, int):
        call_idx = torch.ones([], device=device) * call_idx
    ministep = ((ministep % n_ministeps) / n_ministeps) if n_ministeps > 0 else 0
    if reverse:
        return ((1 - (1 / 1.5 ** (n_calls - call_idx - ministep))) / (1 - res_value)).clamp_(0, 1)
    else:
        
        return (((1 / 1.5 ** (call_idx + ministep)) - res_value) / (1 - res_value)).clamp_(0, 1)

print([blend_factor(i, reverse=False).item() for i in range(n_calls + 1)])
print([blend_factor(i, reverse=True).item() for i in range(n_calls + 1)])

## Initializing / Restoring REGAE state

In [ ]:
from tqdm.autonotebook import trange
from livelossplot import PlotLosses


liveloss = PlotLosses(mode='notebook', groups={'Perceptive Loss': ['dec_loss']})

# Seeding the randomness
RANDOM_SEED=4202
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# Training params
channels = image_patch_shape[0]
n_filter = 64
lat_size = 256
batch_size = 64
n_epochs = 128
len_epoch = n_patches // batch_size

# Building the model
encoder = Encoder(channels, n_filter, patch_size, lat_size).to(device)
decoder = Decoder(channels, n_filter, patch_size, 1, lat_size, True).to(device)
perceptual_loss = VGGPerceptualLoss().to(device)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()),
                             lr=5e-5, betas=(0.5, 0.9), eps=1e-08, weight_decay=0, amsgrad=True)

# Loading checkpoint
checkpoint_filename = './v0_checkpoint.pth'
log_filename = './v0_log_history.pickle'
training_step = 0
running_dec_loss = 0.
if os.path.exists(checkpoint_filename):
    print('checkpoint found, loading...')
    checkpoint = torch.load(checkpoint_filename)
    encoder.load_state_dict(checkpoint['encoder_state_dict'])
    decoder.load_state_dict(checkpoint['decoder_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    training_step = checkpoint['training_step']
    running_dec_loss = checkpoint['dec_loss']

    with open(log_filename, 'rb') as handle:
        log_history = pickle.load(handle)
    
    relog_history = []
    for i in range(len(log_history['dec_loss'])):
        relog_history.append({})
        for k in log_history.keys():
            relog_history[i][k] = np.clip(log_history[k][i].value, 0, 1)

    for log in relog_history:
        liveloss.update(log)

    liveloss.send()

print(f'Encoder params: {count_parameters(encoder)} Decoder params: {count_parameters(decoder)} ')
print(f'Image params: {np.prod(tensor_image_shape)} Compressed Image params: {np.prod((lat_size,) + compressed_shape)}')

torch.cuda.empty_cache()

## Training

In [ ]:
for epoch in range(training_step // len_epoch, n_epochs):    
    batch_mult = (epoch // 64) + 1
    curr_lr = 5e-5 / 2 ** (epoch // 64)
    for g in optimizer.param_groups:
        g['lr'] = curr_lr
    t = trange(training_step % len_epoch, len_epoch, desc=f'Epoch {epoch} | BM: {batch_mult} | LR: {curr_lr}', dynamic_ncols=True)
    for _ in t:
        # zero the parameter gradients
        optimizer.zero_grad()
        
        for m in range(batch_mult):

            # get batch
            image_patch_batch = get_random_patch_batch(batch_size, tensor_image)

            # computing multi-step inputs
            call_idcs = torch.randint(n_calls + 1, (batch_size,), device=device)
            ministeps = torch.rand(batch_size, device=device) * 1000

            bf = blend_factor(call_idcs, ministeps)[:, None, None, None]
            inputs = bf * randinit((batch_size, ) + image_patch_shape) + (1 - bf) * image_patch_batch
            
            lat_code = encoder(image_patch_batch)
            outputs = decoder(inputs, (lat_code,))            
            dec_loss = (1 / batch_mult) * perceptual_loss(outputs, image_patch_batch)
            dec_loss.backward()
        
        # update parameters
        optimizer.step()
        training_step += 1

        # print statistics
        running_dec_loss = 0.9 * running_dec_loss + 0.1 * batch_mult * dec_loss.item()
        
        t.set_postfix({'dec_loss': f'{np.sqrt(running_dec_loss):.6f}'})
        liveloss.update({'dec_loss': np.sqrt(batch_mult * dec_loss.item())})
        
    del image_patch_batch, call_idcs, ministeps, bf, inputs, lat_code, outputs, dec_loss
    torch.cuda.empty_cache()

    torch.save({'encoder_state_dict': encoder.state_dict(),
                'decoder_state_dict': decoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'training_step': training_step,
                'dec_loss': running_dec_loss}, checkpoint_filename)
    
    with open(log_filename, 'wb') as handle:
        pickle.dump(liveloss.logger.log_history, handle, protocol=pickle.HIGHEST_PROTOCOL)

    liveloss.send()

print('Finished Training')


In [ ]:
# del image_patch_batch, call_idcs, ministeps, bf, inputs, lat_code, outputs, dec_loss
# torch.cuda.empty_cache()

In [ ]:
# torch.save({'encoder_state_dict': encoder.state_dict(),
#             'decoder_state_dict': decoder.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'epoch': epoch,
#             'training_step': training_step,
#             'dec_loss': running_dec_loss}, checkpoint_filename)

# with open(log_filename, 'wb') as handle:
#     pickle.dump(liveloss.logger.log_history, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Initializing / Restoring Gencoder

In [ ]:
liveloss = PlotLosses(mode='notebook', groups={'Generative Loss': ['gen_loss_short', 'gen_loss']})

# Seeding the randomness
RANDOM_SEED=4202
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# Training params
# channels = image_patch_shape[0]
# n_filter = 64
# lat_size = 256
genpatch_size = 32
batch_size = 64
n_epochs = 256
len_epoch = n_patches // batch_size # (genpatch_size * batch_size)

gencoder = Decoder(lat_size, lat_size, genpatch_size, 1, n_filter, False, n_filter_factor=1).to(device)
gen_optimizer = torch.optim.Adam(gencoder.parameters(), lr=5e-5, betas=(0.5, 0.9), eps=1e-08, weight_decay=0, amsgrad=True)

# Loading checkpoint
checkpoint_filename = './v0_checkpoint_gen.pth'
log_filename = './v0_log_history_gen.pickle'
training_step = 0
running_gen_loss = 0.
running_gen_loss_short = 0.
if os.path.exists(checkpoint_filename):
    print('checkpoint found, loading...')
    checkpoint = torch.load(checkpoint_filename)
    gencoder.load_state_dict(checkpoint['gencoder_state_dict'])
    gen_optimizer.load_state_dict(checkpoint['gen_optimizer_state_dict'])
    training_step = checkpoint['training_step']
    running_gen_loss = checkpoint['gen_loss']
    running_gen_loss_short = checkpoint['gen_loss_short']

    with open(log_filename, 'rb') as handle:
        log_history = pickle.load(handle)
    
    relog_history = []
    for i in range(len(log_history['gen_loss'])):
        relog_history.append({})
        for k in log_history.keys():
            relog_history[i][k] = log_history[k][i].value

    for log in relog_history:
        liveloss.update(log)

    liveloss.send()

print(f'Gencoder params: {count_parameters(gencoder)}')
print(f'Image params: {np.prod(tensor_image_shape)} Compressed Image params: {np.prod((lat_size,) + compressed_shape)}')

torch.cuda.empty_cache()

In [ ]:
# # Pre-encoding image:
# n_offsets = 4
# embpatch_size = 16
# tensor_image_emb = torch.empty((n_offsets, lat_size, ) + compressed_shape, device='cpu', dtype=torch.float32)
# with torch.no_grad():
#     for o in trange(n_offsets):
#         offset = (patch_size // n_offsets) * o
#         for x_emb in trange(0, compressed_shape[0] - embpatch_size, embpatch_size):
#             for y_emb in trange(0, compressed_shape[1] - embpatch_size, embpatch_size):
#                 x_start = x_emb * patch_size + offset
#                 x_end = (x_emb + embpatch_size) * patch_size + offset
#                 y_start = y_emb * patch_size + offset
#                 y_end = (y_emb + embpatch_size) * patch_size + offset
#                 image_patch_batch = get_patch_any_size(tensor_image, x_start, x_end, y_start, y_end)
#                 lat_code = encoder(image_patch_batch)
#                 lat_code = lat_code.reshape(embpatch_size, embpatch_size, lat_size).permute(2, 0, 1)
#                 tensor_image_emb[o, :, x_emb :x_emb + embpatch_size, y_emb:y_emb + embpatch_size] = lat_code.to('cpu')

# Pre-encoding image:
n_offsets = 4
embpatch_size = 16

tensor_image_emb_filename = 'v0_tensor_image_emb.pth'
if os.path.exists(tensor_image_emb_filename):
    print('tensor_image_emb found, loading...')
    tensor_image_emb = torch.load(tensor_image_emb_filename)['tensor_image_emb']
else:
    tensor_image_emb = torch.zeros((n_offsets, lat_size, ) + compressed_shape, device='cpu', dtype=torch.float32)
    with torch.no_grad():
        for o in trange(n_offsets):
            offset = o / n_offsets
            for x in trange(compressed_shape[0] - (1 if o > 0 else 0)):
                for y in range(compressed_shape[1] - (1 if o > 0 else 0)):
                    image_patch = get_image_patch(tensor_image, patch_size, (x + offset) / compressed_shape[0], (y + offset) / compressed_shape[1])
                    tensor_image_emb[o, :, x, y] = encoder(image_patch[None, ...])[0].to('cpu')
    
    torch.save({'tensor_image_emb': tensor_image_emb}, tensor_image_emb_filename)

print(tensor_image_emb.shape)

## Training Gencoder

In [ ]:
for epoch in range(training_step // len_epoch, n_epochs):    
    batch_mult = (epoch // 128) + 1
    curr_lr = 5e-5 / 2 ** (epoch // 64)
    for g in gen_optimizer.param_groups:
        g['lr'] = curr_lr
    t = trange(training_step % len_epoch, len_epoch, desc=f'Epoch {epoch} | BM: {batch_mult} | LR: {curr_lr}', dynamic_ncols=True)
    for _ in t:
        # zero the parameter gradients
        gen_optimizer.zero_grad()
        
        for m in range(batch_mult * 2):
            
            with torch.no_grad():
                lat_code_batch = []
                
                # for _ in range(batch_size):
                #     x_start = np.clip(np.random.rand() * tensor_image_shape[1], 0, tensor_image_shape[1] - genpatch_size * patch_size).astype(int)
                #     x_end = x_start + genpatch_size * patch_size
                #     y_start = np.clip(np.random.rand() * tensor_image_shape[2], 0, tensor_image_shape[2] - genpatch_size * patch_size).astype(int)
                #     y_end = y_start + genpatch_size * patch_size
                #     # get batch
                #     image_patch_batch = get_patch_any_size(tensor_image, x_start, x_end, y_start, y_end)
                #     lat_code = encoder(image_patch_batch)
                #     lat_code_batch.append(lat_code)
                # lat_code_batch = torch.stack(lat_code_batch)
                # lat_code_batch = torch.permute(lat_code_batch, (0, 2, 1)).reshape(batch_size, lat_size, genpatch_size, genpatch_size)

                for o in range(n_offsets):
                    for _ in range(batch_size // n_offsets):
                        x = np.random.randint(compressed_shape[0] - genpatch_size - (1 if o > 0 else 0))
                        y = np.random.randint(compressed_shape[1] - genpatch_size - (1 if o > 0 else 0))
                        lat_code_batch.append(tensor_image_emb[o, :, x:x + genpatch_size, y:y + genpatch_size])
                lat_code_batch = torch.stack(lat_code_batch).to(device)

            lat_code_batch.requires_grad_(True)

            # computing multi-step inputs
            call_idcs = torch.randint(n_calls + 1, (batch_size,), device=device)
            ministeps = torch.rand(batch_size, device=device) * 1000
            lat_init = randinit(lat_code_batch.shape)

            bf = blend_factor(call_idcs, ministeps)[:, None, None, None]
            inputs = bf * lat_init + (1 - bf) * lat_code_batch

            if m % 2 == 0:
                bf_next = blend_factor(call_idcs + 1, ministeps)[:, None, None, None]
                outputs_target = bf_next * lat_init + (1 - bf_next) * lat_code_batch
                
                outputs = gencoder(inputs)
                gen_loss = (1 / batch_mult) * F.l1_loss(outputs, outputs_target)
                gen_loss.backward()      
            else:                
                outputs = gencoder(inputs, (torch.ones((batch_size, n_filter), device=device), ))
                gen_loss_short = (1 / batch_mult) * F.l1_loss(outputs, lat_code_batch)
                gen_loss_short.backward()
        
        # update parameters
        gen_optimizer.step()
        training_step += 1

        # print statistics
        running_gen_loss = 0.9 * running_gen_loss + 0.1 * batch_mult * gen_loss.item()
        running_gen_loss_short = 0.9 * running_gen_loss_short + 0.1 * batch_mult * gen_loss_short.item()
        
        t.set_postfix({'gen_loss': f'{np.sqrt(running_gen_loss):.6f}', 'gen_loss_short': f'{np.sqrt(running_gen_loss_short):.6f}'})
        liveloss.update({'gen_loss': np.sqrt(batch_mult * gen_loss.item()), 'gen_loss_short': np.sqrt(batch_mult * gen_loss_short.item())})
        
    del lat_code_batch, call_idcs, ministeps, bf, inputs, outputs, gen_loss, gen_loss_short
    torch.cuda.empty_cache()

    torch.save({'gencoder_state_dict': gencoder.state_dict(),
                'gen_optimizer_state_dict': gen_optimizer.state_dict(),
                'epoch': epoch,
                'training_step': training_step,
                'gen_loss': running_gen_loss,
                'gen_loss_short': running_gen_loss_short}, checkpoint_filename)
    
    with open(log_filename, 'wb') as handle:
        pickle.dump(liveloss.logger.log_history, handle, protocol=pickle.HIGHEST_PROTOCOL)

    liveloss.send()

In [ ]:
del lat_code_batch, call_idcs, ministeps, bf, inputs, outputs, gen_loss, gen_loss_short
torch.cuda.empty_cache()

In [ ]:
torch.save({'gencoder_state_dict': gencoder.state_dict(),
            'gen_optimizer_state_dict': gen_optimizer.state_dict(),
            'epoch': epoch,
            'training_step': training_step,
            'gen_loss': running_gen_loss,
            'gen_loss_short': running_gen_loss_short}, checkpoint_filename)

with open(log_filename, 'wb') as handle:
    pickle.dump(liveloss.logger.log_history, handle, protocol=pickle.HIGHEST_PROTOCOL)


## Testing

In [ ]:
import matplotlib.pyplot as plt

def plot_patch(image_patch, x_start, x_end, y_start, y_end, patch_size):
    image_patch_rec = reconstruct_image_from_patch_batch(image_patch, (x_end - x_start) // patch_size, (y_end - y_start) // patch_size)
    image_patch_rec = (image_patch_rec + 1.) / 2
    plt.imshow(image_patch_rec.to('cpu').permute(1,2,0))

In [ ]:
x_start = 2280
x_end = 2280 + 1024
y_start = 2280
y_end = 2280 + 1024
with torch.no_grad():
    image_patch = get_patch_any_size(tensor_image, x_start, x_end, y_start, y_end)
    plot_patch(image_patch, x_start, x_end, y_start, y_end, patch_size)

In [ ]:
with torch.no_grad():    
    decompressed_patch = randinit((image_patch.shape[0], ) + image_patch_shape)
    n_batches = image_patch.shape[0] // batch_size
    for b in range(n_batches):
        lat_code = encoder(image_patch[b * batch_size: (b + 1) * batch_size])
        # for n in range(n_calls):
        #     bf = blend_factor(n)
        #     decompressed_patch[b * batch_size: (b + 1) * batch_size] = bf * randinit((batch_size, ) + image_patch_shape) + (1 - bf) * decompressed_patch[b * batch_size: (b + 1) * batch_size]
        #     decompressed_patch[b * batch_size: (b + 1) * batch_size] = requantize(decoder(decompressed_patch[b * batch_size: (b + 1) * batch_size], (lat_code, )))
        decompressed_patch[b * batch_size: (b + 1) * batch_size] = requantize(decoder(decompressed_patch[b * batch_size: (b + 1) * batch_size], (lat_code, )))
    
    plot_patch(decompressed_patch, x_start, x_end, y_start, y_end, patch_size)

In [ ]:
decomp_error = (image_patch - decompressed_patch)
#decomp_error = (decomp_error - decomp_error.mean()) / decomp_error.std()
plot_patch(decomp_error, x_start, x_end, y_start, y_end, patch_size)
decomp_error.abs().mean()

In [ ]:
with torch.no_grad():
    lat_code_batch = get_image_patch(tensor_image_emb[0], genpatch_size, (x_start/patch_size) / compressed_shape[0], (y_start/patch_size) / compressed_shape[1])
    lat_code_batch = lat_code_batch.reshape(lat_size, genpatch_size * genpatch_size).permute(1, 0)
    decompressed_patch = randinit((image_patch.shape[0], ) + image_patch_shape)
    n_batches = image_patch.shape[0] // batch_size
    for b in range(n_batches):
        lat_code = lat_code_batch[b * batch_size: (b + 1) * batch_size]
        decompressed_patch[b * batch_size: (b + 1) * batch_size] = requantize(decoder(decompressed_patch[b * batch_size: (b + 1) * batch_size], (lat_code, )))
    
    plot_patch(decompressed_patch, x_start, x_end, y_start, y_end, patch_size)

In [ ]:
def plot_lat_code_batch(lat_code_batch, x_start, x_end, y_start, y_end, patch_size):
    lat_code_batch_rec = lat_code_batch[0].reshape(lat_size, genpatch_size * genpatch_size).permute(1, 0)
    decompressed_patch = randinit((image_patch.shape[0], ) + image_patch_shape)
    n_batches = image_patch.shape[0] // batch_size
    for b in range(n_batches):
        lat_code = lat_code_batch_rec[b * batch_size: (b + 1) * batch_size]
        decompressed_patch[b * batch_size: (b + 1) * batch_size] = requantize(decoder(decompressed_patch[b * batch_size: (b + 1) * batch_size], (lat_code, )))
    
    # decompressed_patch[:, :, :, -1] = 1.
    # decompressed_patch[:, :, -1, :] = 1.
    plt.figure()
    plot_patch(decompressed_patch, x_start, x_end, y_start, y_end, patch_size)

In [ ]:
with torch.no_grad():
    #lat_code_batch = get_image_patch(tensor_image_emb[0], genpatch_size, (x_start/patch_size) / compressed_shape[0], (y_start/patch_size) / compressed_shape[1])[None, ...]
    lat_code_batch = randinit((1, lat_size, genpatch_size, genpatch_size))
    # plot_lat_code_batch(lat_code_batch, x_start, x_end, y_start, y_end, patch_size)
    # print('initial noise')
    for n in range(n_calls * 2):
        bf = blend_factor(n, n_calls=n_calls * 2)
        lat_code_batch = bf * randinit((1, lat_size, genpatch_size, genpatch_size)) + (1 - bf) * lat_code_batch
        
        lat_code_batch = gencoder(lat_code_batch)
        #plot_lat_code_batch(lat_code_batch, x_start, x_end, y_start, y_end, patch_size)
        # print('step: ', n)
    
        lat_code_batch = gencoder(lat_code_batch, (torch.ones((1, n_filter), device=device), ))
        # plot_lat_code_batch(lat_code_batch, x_start, x_end, y_start, y_end, patch_size)
        # print('+ shortcut')
    
    plot_lat_code_batch(lat_code_batch, x_start, x_end, y_start, y_end, patch_size)

In [ ]:
with torch.no_grad():
    n_calls_map = n_calls * torch.ones(image_patch_rec.shape[1:], dtype=torch.float32, device=device)
    # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
    # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
    generated_patch = randinit((1,) + image_patch_rec.shape)
    x_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat_interleave(patch_size)
    y_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat(patch_size)
    t = trange(n_calls - 1, -1, -1, desc='Generating image', dynamic_ncols=True)
    for c in t:
        while torch.any(n_calls_map > c).item():
            # target_coords = (n_calls_map_soft > c).nonzero()
            # if target_coords.shape[0] == 0:
            target_coords = (n_calls_map > c).nonzero()
                
            rand_coord = torch.randint(target_coords.shape[0], ())
            x_coords = target_coords[rand_coord][0].clamp(patch_size // 2, image_patch_rec.shape[1] - patch_size // 2) + x_frame
            y_coords = target_coords[rand_coord][1].clamp(patch_size // 2, image_patch_rec.shape[2] - patch_size // 2) + y_frame
            
        
            generated_patch_tmp = generated_patch[:, :, x_coords, y_coords].view(1, 3, patch_size, patch_size)
            bf = blend_factor(n_calls - 1 - c)
            generated_patch_tmp = bf * (1 - 1e-2) * randinit(generated_patch_tmp.shape) + (1 - bf) * generated_patch_tmp

            #noise_mask = (n_calls_map[x_coords, y_coords] < 1).view(1, 1, patch_size, patch_size).to(torch.float32)
            #generated_patch_tmp += noise_mask * blend_factor(n_calls - 1) * torch.rand_like(generated_patch_tmp)
            
            glat_code = gencoder(generated_patch_tmp)
            
            update_mask = (n_calls_map[x_coords, y_coords] > c).view(1, 1, -1).to(torch.float32)
            generated_patch[:, :, x_coords, y_coords] = (1 - update_mask) * generated_patch[:, :, x_coords, y_coords] + update_mask * requantize(decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size))
            
            #generated_patch[:, :, x_coords, y_coords] = decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size)

            n_calls_map[x_coords, y_coords] = (1 - update_mask[0, 0]) * n_calls_map[x_coords, y_coords] + update_mask[0, 0] * (n_calls_map[x_coords, y_coords] - 1)
            t.set_postfix({'min call': f'{n_calls_map.min():.2f}', 'max call': f'{n_calls_map.max():.2f}', 'average call': f'{n_calls_map.mean():.2f}'})
                
            # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
            # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
        
    generated_patch_rec = (generated_patch[0] + 1.) / 2
    plt.imshow(generated_patch_rec.to('cpu').permute(1,2,0))

In [ ]:
with torch.no_grad():
    n_calls_map = n_calls * torch.ones(image_patch_rec.shape[1:], dtype=torch.float32, device=device)
    # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
    # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
    generated_patch = torch.rand((1,) + image_patch_rec.shape, device=device)
    x_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat_interleave(patch_size)
    y_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat(patch_size)
    t = trange(n_calls - 1, -1, -1, desc='Generating image', dynamic_ncols=True)
    for c in t:
        while torch.any(n_calls_map > c).item():
            # target_coords = (n_calls_map_soft > c).nonzero()
            # if target_coords.shape[0] == 0:
            target_coords = (n_calls_map > c).nonzero()
                
            rand_coord = torch.randint(target_coords.shape[0], ())
            x_coords = target_coords[rand_coord][0].clamp(patch_size // 2, image_patch_rec.shape[1] - patch_size // 2) + x_frame
            y_coords = target_coords[rand_coord][1].clamp(patch_size // 2, image_patch_rec.shape[2] - patch_size // 2) + y_frame
            
        
            generated_patch_tmp = generated_patch[:, :, x_coords, y_coords].view(1, 3, patch_size, patch_size)
            
            #noise_mask = (n_calls_map[x_coords, y_coords] < 1).view(1, 1, patch_size, patch_size).to(torch.float32)
            #generated_patch_tmp += noise_mask * blend_factor(n_calls - 1) * torch.rand_like(generated_patch_tmp)
            
            glat_code = gencoder(generated_patch_tmp)
            
            # update_mask = (n_calls_map[x_coords, y_coords] > 0).view(1, 1, -1).to(torch.float32)
            # generated_patch[:, :, x_coords, y_coords] = (1-update_mask) * generated_patch[:, :, x_coords, y_coords] + update_mask * decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size)
            
            generated_patch[:, :, x_coords, y_coords] = decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size)

            n_calls_map[x_coords, y_coords] -= 1
            t.set_postfix({'min call': f'{n_calls_map.min():.2f}', 'max call': f'{n_calls_map.max():.2f}', 'average call': f'{n_calls_map.mean():.2f}'})
                
            # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
            # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
        
    generated_patch_rec = generated_patch[0].clamp_(0, 1)
    plt.imshow(generated_patch_rec.to('cpu').permute(1,2,0))

In [ ]:
with torch.no_grad():
    n_calls_map = n_calls * torch.ones(image_patch_rec.shape[1:], dtype=torch.float32, device=device)
    # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
    # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
    generated_patch = torch.rand((1,) + image_patch_rec.shape, device=device)
    x_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat_interleave(patch_size)
    y_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat(patch_size)
    t = trange(n_calls - 1, -1, -1, desc='Generating image', dynamic_ncols=True)
    for c in t:
        while torch.any(n_calls_map > c).item():
            # target_coords = (n_calls_map_soft > c).nonzero()
            # if target_coords.shape[0] == 0:
            target_coords = (n_calls_map > c).nonzero()
                
            rand_coord = torch.randint(target_coords.shape[0], ())
            x_coords = target_coords[rand_coord][0].clamp(patch_size // 2, image_patch_rec.shape[1] - patch_size // 2) + x_frame
            y_coords = target_coords[rand_coord][1].clamp(patch_size // 2, image_patch_rec.shape[2] - patch_size // 2) + y_frame
            
            for _ in range(n_calls):
                generated_patch_tmp = generated_patch[:, :, x_coords, y_coords].view(1, 3, patch_size, patch_size)
                
                #noise_mask = (n_calls_map[x_coords, y_coords] < 1).view(1, 1, patch_size, patch_size).to(torch.float32)
                #generated_patch_tmp += noise_mask * blend_factor(n_calls - 1) * torch.rand_like(generated_patch_tmp)
                
                glat_code = gencoder(generated_patch_tmp)
                
                update_mask = (n_calls_map[x_coords, y_coords] > 0).view(1, 1, -1).to(torch.float32)
                generated_patch[:, :, x_coords, y_coords] = (1-update_mask) * generated_patch[:, :, x_coords, y_coords] + update_mask * decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size)
                
                #generated_patch[:, :, x_coords, y_coords] = decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size)

                n_calls_map[x_coords, y_coords] -= 1
                t.set_postfix({'min call': f'{n_calls_map.min():.2f}', 'max call': f'{n_calls_map.max():.2f}', 'average call': f'{n_calls_map.mean():.2f}'})
                
            # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
            # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
        
    generated_patch_rec = generated_patch[0].clamp_(0, 1)
    plt.imshow(generated_patch_rec.to('cpu').permute(1,2,0))

In [ ]:
plt.imshow(((n_calls_map / n_calls_map.min() < 0.1).to(torch.float32) * generated_patch_rec).to('cpu').permute(1,2,0))

In [ ]:
with torch.no_grad():
    n_calls_map = n_calls * torch.ones(image_patch_rec.shape[1:], dtype=torch.float32, device=device)
    # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
    # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
    generated_patch = torch.rand((1, 3) + (image_patch_rec.shape[1] + 4 * patch_size, image_patch_rec.shape[2] + 4 * patch_size), device=device)
    x_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat_interleave(patch_size)
    y_frame = torch.arange(-(patch_size // 2), (patch_size // 2), device=device).repeat(patch_size)
    t = trange(n_calls - 1, -1, -1, desc='Generating image', dynamic_ncols=True)
    for c in t:
        for p_x in range(2 * patch_size, generated_patch.shape[2] - 2 * patch_size, patch_size):
            for p_y in range(2 * patch_size, generated_patch.shape[2] - 2 * patch_size, patch_size):
                x_coords = p_x + (n_calls - c) * (patch_size // n_calls) + x_frame
                y_coords = p_y + (n_calls - c) * (patch_size // n_calls) + y_frame
            
                generated_patch_tmp = generated_patch[:, :, x_coords, y_coords].view(1, 3, patch_size, patch_size)
                
                #noise_mask = (n_calls_map[x_coords, y_coords] < 1).view(1, 1, patch_size, patch_size).to(torch.float32)
                #generated_patch_tmp += noise_mask * blend_factor(n_calls - 1) * torch.rand_like(generated_patch_tmp)
                
                glat_code = gencoder(generated_patch_tmp)
                
                update_mask = (n_calls_map[x_coords, y_coords] > 0).view(1, 1, -1).to(torch.float32)
                generated_patch[:, :, x_coords, y_coords] = (1-update_mask) * generated_patch[:, :, x_coords, y_coords] + update_mask * decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size)
                
                #generated_patch[:, :, x_coords, y_coords] = decoder(generated_patch_tmp, (None, glat_code)).view(1, 3, patch_size * patch_size)

                n_calls_map[x_coords, y_coords] -= 1
                t.set_postfix({'min call': f'{n_calls_map.min():.2f}', 'max call': f'{n_calls_map.max():.2f}', 'average call': f'{n_calls_map.mean():.2f}'})
            
        # n_calls_map_soft = F.interpolate(n_calls_map[None, None, ...], scale_factor=1 / 32, mode='bilinear')
        # n_calls_map_soft = F.interpolate(n_calls_map_soft, scale_factor=32, mode='bilinear')[0, 0]
        
    generated_patch_rec = generated_patch[0, :, 2 * patch_size:-2 * patch_size, 2 * patch_size:-2 * patch_size].clamp_(0, 1)
    plt.imshow(generated_patch_rec.to('cpu').permute(1,2,0))

In [ ]:
del image_patch, lat_code, decompressed_patch, lat_code_batch#, generated_patch
torch.cuda.empty_cache()